<a href="https://colab.research.google.com/github/Bryce-Neuman/GB-885-Final-Project-Neuman-Bryce-T/blob/main/GB885_Final_Project_Neuman_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You work as a sales analyst for RUSH, a globally renowned sportswear and footwear brand known for its innovative designs and performance-oriented products. The company stores its raw sales data as a collection of three tables:

TABLE_PRODUCTS
TABLE_RETAILER
TABLE_SALES
The data includes the number of units sold, the total sales revenue, the location of the sales, the type of product sold, as well as other relevant information. (For data field definitions and explanations, see the data dictionary.) The data is "raw," meaning it has not been cleaned and probably contains errors that need to be addressed.

The VP of US Sales has tasked you with analyzing sales data for trends and insights that will help company leadership understand the market and identify opportunities for growth. For example, you may want to look for trends or insights in seasonality, retailers, locations, or sales methods. Take initiative to apply your creativity and curiosity to this data.

In addition, she has asked you to answer the following business questions:

What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
What state had the highest sales (in dollars) of women's products in 2021? How much was it?
What state had the highest sales (in dollars) of men's products in 2021? How much was it?
What retailer purchased the most units in 2021? In 2020?

In [ ]:
# Load the required code libraries.

import pandas as pd
import numpy
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
!git clone https://github.com/Bryce-Neuman/GB-885-Final-Project-Neuman-Bryce-T

In [ ]:
import pandas as pd

# Load the data (CSV files from github)
#Table_Products_885 is a pipe delimited table so it has the pipe delimiter noted otherwise the table will not populate correctly
df_products = pd.read_csv("GB-885-Final-Project-Neuman-Bryce-T/TABLE_PRODUCTS_885.csv", sep='|')
df_retailer = pd.read_csv("GB-885-Final-Project-Neuman-Bryce-T/TABLE_RETAILER_885.csv")
df_sales = pd.read_csv("GB-885-Final-Project-Neuman-Bryce-T/TABLE_SALES_885.csv")

Preview some data from df_products

In [ ]:

df_products.head()

Preview some data from df_retailer

In [ ]:
df_retailer.head()

Preview some data from df_sales

In [ ]:
df_sales.head()

Clean and ensure DF_Products is ready to consume

In [ ]:
#Check the size of the dataframe
df_products.shape

Dataframe is only 6x2. We could run statistics on it but it seems fine to skip and just view the whole table.

In [ ]:
#Display the 6 rows in df_products
df_products.head(6)

No issues seem present in df_products. Next check df_retailer

In [ ]:
#Check the size of df_retailer
df_retailer.shape

Since this is 110 rows with 5 columns, we will look at statistics and check data with code vs manually reviewing

In [ ]:
#Check statistics of df_retailer
df_retailer.describe()

THere appears to be a duplicate retailer_id as there are 110 total but 106 unique

In [ ]:
#count how many nulls are present
df_retailer.isnull().sum()

No nulls are present.

In [ ]:
#Get rid of the duplicate retailer ID's
df_retailer.drop_duplicates(subset='RETAILER_ID', keep='first', inplace=True)

# Verify the changes
print("Shape after dropping duplicates:", df_retailer.shape)
display(df_retailer.describe())

Check df_sales

In [ ]:
df_sales.shape

In [ ]:
#Check statistics. This is too big to read through them all
df_sales.describe()

Price per unit of 99999 seems like a place holder as these are talking about clothes so that seems very unlikely especially with the 75% being 55.



In [ ]:
# I am going to get rid of any values that are over 1000
df_sales=df_sales[df_sales['PRICE_PER_UNIT']<=1000].copy()

In [ ]:
#recheck the statistics to ensure its resolved
df_sales.describe()

In [ ]:
#count how many nulls are present
df_sales.isnull().sum()

In [ ]:
df_sales.dtypes

In [ ]:
#Change the column units_sold to an int64 so that we can do math on it this also requires getting rid of some values that were not numbers otherwise it caused an error.
df_sales['UNITS_SOLD'] = pd.to_numeric(df_sales['UNITS_SOLD'], errors='coerce')
df_sales = df_sales.dropna(subset=['UNITS_SOLD'])
df_sales['UNITS_SOLD'] = df_sales['UNITS_SOLD'].astype('int64')

##With no nulls present and all tables cleaned, being to answer the business questions

What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?

What state had the highest sales (in dollars) of women's products in 2021? How much was it?

What state had the highest sales (in dollars) of men's products in 2021? How much was it?

What retailer purchased the most units in 2021? In 2020?

#Question 1

What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?

In [ ]:
#Create a new column for revenue that is price per unit multiplied by the units sold to get dollars of revenue
df_sales['revenue'] = df_sales['PRICE_PER_UNIT'] * df_sales['UNITS_SOLD']

sales_2021 = (
    df_sales[df_sales['YEAR'] == 2021]
    .groupby('PRODUCT_ID')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)

Sales_2021_w_prod=sales_2021.merge(df_products, on='PRODUCT_ID', how='left')

print(Sales_2021_w_prod.head)

Top product ID is 20 which corresponds to mens street footwear

In [ ]:
#Plot the above to make a chart
plt.figure(figsize=(10, 6))
sns.barplot(data=Sales_2021_w_prod, x='revenue', y='PRODUCT_NAME', hue='PRODUCT_NAME', legend=False)
plt.title('Total 2021 Sales by Product')
plt.xlabel('Revenue ($)')
plt.ylabel('Product')

# Format x-axis as readable dollar amounts instead of scientific notation
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'${x/1e6:.1f}M'))


plt.tight_layout()
plt.show()

Create a dataframe that contains all 3 tables to make future questions easier along with future investigation if needed.

In [ ]:
# Merge df_sales with df_products
df_merged = pd.merge(df_sales, df_products, on='PRODUCT_ID', how='left')

# Merge the result with df_retailer
df_RUSH_business = pd.merge(df_merged, df_retailer, on='RETAILER_ID', how='left')

# Display the first few rows of the merged DataFrame to verify
display(df_RUSH_business.head())

#Question 2

What state had the highest sales (in dollars) of women's products in 2021? How much was it?


In [ ]:
#Create a dataframe specifically for sales in the year 2021 where the word women is in the product name and group by state and revenue with the highest values first. Pick just the top 10 so this can be easily graphed later
Womens_sales_2021 = (
    df_RUSH_business[
        (df_RUSH_business['YEAR'] == 2021) &
        (df_RUSH_business['PRODUCT_NAME'].str.contains('women', case=False, na=False))
    ]
    .groupby('STATE')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
    .head(10)
)


print(Womens_sales_2021.head)

In [ ]:
#Plot the above to make a chart
plt.figure(figsize=(10, 6))
sns.barplot(Womens_sales_2021, x='revenue', y='STATE', hue='STATE', legend=False)
plt.title('Total 2021 Womens Sales by Top 10 States')
plt.xlabel('Revenue ($)')
plt.ylabel('STATE')

plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

# Question 3

What state had the highest sales (in dollars) of men's products in 2021? How much was it?

In [ ]:
#Create a dataframe specifically for sales in the year 2021 where the word women is NOT in the product name and group by state and revenue with the highest values first. Pick just the top 10 so this can be easily graphed later
#Since this just has mens and womesn and not childrens or any other category, this code is just looking where it doesnt contain women as if I switch the code above to men, women still gets a hit.
Mens_sales_2021 = (
    df_RUSH_business[
        (df_RUSH_business['YEAR'] == 2021) &
        (~df_RUSH_business['PRODUCT_NAME'].str.contains('women', case=False, na=False))
    ]
    .groupby('STATE')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
    .head(10)
)


print(Mens_sales_2021.head)

In [ ]:
#Plot the above to make a chart
plt.figure(figsize=(10, 6))
sns.barplot(Mens_sales_2021, x='revenue', y='STATE', hue='STATE', legend=False)
plt.title('Total 2021 Mens Sales by Top 10 States')
plt.xlabel('Revenue ($)')
plt.ylabel('STATE')

plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

#Question 4

What retailer purchased the most units in 2021? In 2020?

In [ ]:
#This query will look at not only the retailer with the most but the top 5 retailers per year. This is likely more useful as we can see if what was first is now 2nd or 3rd or drops off all together.

top3_retailer_by_year = (
    df_RUSH_business.groupby(['YEAR', 'RETAILER'], as_index=False)['UNITS_SOLD']
      .sum()
      .sort_values(['YEAR', 'UNITS_SOLD'], ascending=[True, False])
      .groupby('YEAR')
      .head(3)
)

top3_retailer_by_year

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top3_retailer_by_year, x='YEAR', y='UNITS_SOLD', hue='RETAILER')
plt.title('Top 3 Retailers by Units Sold, Per Year')
plt.xlabel('Year')
plt.ylabel('Units Sold')
plt.legend(title='Retailer', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, pos: f'${y/1e6:.1f}M'))

plt.tight_layout()
plt.show()